
# Storing and Querying Embeddings Using ChromaDB

## Objective

Learn how to store, persist, and retrieve text embeddings using ChromaDB.

We will use Google's Gemini embedding model to generate vectors and ChromaDB to perform similarity search.

### Learning Objectives

1. Understand what a vector database stores.
2. Create a persistent ChromaDB database.
3. Create a collection.
4. Store documents alongside their embeddings.
5. Perform semantic similarity searches.


In [69]:

import os
import chromadb

from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise RuntimeError("GOOGLE_API_KEY not found in environment.")

client = genai.Client(api_key=api_key)

print("Gemini and ChromaDB initialized successfully.")


Gemini and ChromaDB initialized successfully.



## 1. Create a Persistent ChromaDB Database

ChromaDB is a vector database that stores embeddings and their associated documents.

A ChromaDB collection is similar to a table in a relational database. It groups related documents and vectors.

We use a persistent client so our data remains stored on disk even after restarting the Jupyter kernel.


In [70]:

# Create a persistent ChromaDB database
chroma_client = chromadb.PersistentClient(
    path="./chroma_db"
)

# Create or retrieve a collection
collection = chroma_client.get_or_create_collection(
    name="codeDeck-chroma-db-demo",
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB collection created successfully!")
print("Collection name:", collection.name)


ChromaDB collection created successfully!
Collection name: codeDeck-chroma-db-demo



## 2. Store Embeddings in ChromaDB

A vector database stores numerical embeddings along with their associated documents and unique identifiers.

In this experiment, we will:

1. Define eight sentences.
2. Generate 768-dimensional embeddings using Gemini.
3. Assign a unique ID to each sentence.
4. Store the embeddings, documents, and IDs in ChromaDB.

The stored embeddings can later be retrieved using semantic similarity search.


In [71]:

texts = [
    "How do I write a Python function?",
    "How can I create a class in Java?",
    "Which smartphone has the best camera?",
    "What is the best Android phone?",
    "Turn off the lights.",
    "Switch the lights off.",
    "Apple released a new iPhone.",
    "Apple is my favorite fruit."
]

print("Total documents:", len(texts))


Total documents: 8


In [72]:

result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=texts,
    config=types.EmbedContentConfig(
        task_type="SEMANTIC_SIMILARITY",
        output_dimensionality=768
    )
)

embeddings = []
ids = []

for i, embedding_obj in enumerate(result.embeddings):
    embeddings.append(embedding_obj.values)
    ids.append(f"id_{i}")

print("Embeddings generated:", len(embeddings))
print("Embedding dimensions:", len(embeddings[0]))
print("Document IDs:", ids)


Embeddings generated: 8
Embedding dimensions: 768
Document IDs: ['id_0', 'id_1', 'id_2', 'id_3', 'id_4', 'id_5', 'id_6', 'id_7']



## 3. Insert Embeddings into ChromaDB

We use the `upsert()` method to store our documents and embeddings.

Each record contains:

- **ID:** Unique identifier for the document.
- **Document:** Original natural-language text.
- **Embedding:** Numerical vector generated by Gemini.

Unlike `add()`, `upsert()` inserts new records or updates existing records with matching IDs.

This makes the notebook safe to rerun without creating duplicate records.


In [73]:

collection.upsert(
    ids=ids,
    documents=texts,
    embeddings=embeddings
)

print("Embeddings stored successfully!")
print("Total documents in collection:", collection.count())


Embeddings stored successfully!
Total documents in collection: 8


In [74]:

stored_data = collection.get(
    include=["documents"]
)

for doc_id, document in zip(
    stored_data["ids"],
    stored_data["documents"]
):
    print(f"{doc_id}: {document}")


id_0: How do I write a Python function?
id_1: How can I create a class in Java?
id_2: Which smartphone has the best camera?
id_3: What is the best Android phone?
id_4: Turn off the lights.
id_5: Switch the lights off.
id_6: Apple released a new iPhone.
id_7: Apple is my favorite fruit.



## 4. Query Embeddings in ChromaDB

### What is Semantic Search?

Semantic search retrieves documents based on their meaning rather than relying only on exact keyword matches.

In this experiment, we will:

1. Accept a new natural-language query.
2. Convert the query into a 768-dimensional embedding using Gemini.
3. Search the ChromaDB collection for the nearest embeddings.
4. Retrieve the most semantically relevant documents.

We use the same embedding model, task type, and dimensionality for both stored documents and queries.


In [75]:

query = "Object Oriented"

query_embedding = client.models.embed_content(
    model="gemini-embedding-001",
    contents=query,
    config=types.EmbedContentConfig(
        task_type="SEMANTIC_SIMILARITY",
        output_dimensionality=768
    )
).embeddings[0].values

print("Query:", query)
print("Query embedding dimensions:", len(query_embedding))


Query: Object Oriented
Query embedding dimensions: 768


In [76]:

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2,
    include=["documents", "distances"]
)

print("Query:", query)
print("Retrieved documents:", results["documents"])
print("Distances:", results["distances"])


Query: Object Oriented
Retrieved documents: [['How can I create a class in Java?', 'How do I write a Python function?']]
Distances: [[0.19378012418746948, 0.2196662425994873]]
